# Flight Delay Data Quality Profile

This notebook uses **PySpark** to assess the quality of the **typed raw Parquet** data before proceeding to feature engineering and modeling.

### Objectives
- verify the schema and data size;
- measure missing and blank values;
- check uniqueness, validity, and cardinality;
- check referential integrity with lookup tables;
- verify the target, eligibility, and leakage;
- summarize issues to hand off to the next module.

### Not performed in this notebook
- imputation or dropping missing values;
- outlier handling;
- airport-code mapping;
- feature engineering;
- train/test splitting;
- model training.

> **Note:** Typed Parquet is still **typed raw data**, not a cleaned or model-ready dataset.

## 1. Runtime and Spark Configuration

Before profiling the data, verify that the notebook is running in the correct Python/PySpark environment and can access the typed Parquet files.

In [ ]:
import os
import sys
from pathlib import Path

import pyspark
from pyspark.sql import SparkSession, functions as F

In [ ]:
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("FlightDataQualityProfile")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "12")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [ ]:
print("Runtime")
print("-" * 40)
print(f"Python  : {sys.version.split()[0]}")
print(f"PySpark : {pyspark.__version__}")
print(f"Spark   : {spark.version}")
print(f"Python executable: {sys.executable}")

### Data source

The notebook reads directly from the typed Parquet files created by
`scripts/prepare_data.py`.


In [ ]:
from pathlib import Path
import os

# 1. Prefer the environment variable if available
env_path = os.environ.get("FLIGHT_TYPED_PARQUET_DIR")

if env_path:
    data_root = Path(env_path)

else:
    # 2. If no environment variable is set, search cwd and nearby parent directories for the repository root
    current = Path.cwd()

    repo_root = None

    for candidate in [current, *current.parents]:
        if (
            (candidate / "requirements.txt").is_file()
            and (candidate / "data").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            repo_root = candidate
            break

    if repo_root is None:
        raise RuntimeError("Cannot locate repository root.")

    data_root = repo_root / "data" / "typed-parquet"


paths = {
    "flights": data_root / "flights",
    "airports": data_root / "airports",
    "airlines": data_root / "airlines",
}

missing = [
    name
    for name, path in paths.items()
    if not path.is_dir()
]

if missing:
    raise FileNotFoundError(
        f"Missing typed Parquet datasets: {missing}"
    )

print("Data source check")
print("-" * 40)
print("Status   : PASS")
print("Datasets : flights, airports, airlines")

## 2. Load Typed Parquet and Inventory the Data

This section reads the three typed Parquet tables and verifies the basic size of the data:

- `flights`: the main table containing flight information;
- `airports`: the airport lookup table;
- `airlines`: the airline lookup table.

The goal is to confirm that the data can be read and that the row counts match the `prepare_data.py` step.

In [ ]:
flights = spark.read.parquet(str(paths["flights"]))
airports = spark.read.parquet(str(paths["airports"]))
airlines = spark.read.parquet(str(paths["airlines"]))

print("Typed Parquet loaded successfully.")

In [ ]:
inventory_data = [
    ("flights", flights.count(), len(flights.columns)),
    ("airports", airports.count(), len(airports.columns)),
    ("airlines", airlines.count(), len(airlines.columns)),
]

inventory_df = spark.createDataFrame(
    inventory_data,
    ["dataset", "rows", "columns"]
)

inventory_df.show(truncate=False)

### Schema and Sample Data

The schema helps verify that numeric columns were cast to the correct types in typed Parquet.

Only a few sample rows are displayed to inspect the data structure; this sample is not used to evaluate a model.

In [ ]:
flights.printSchema()

In [ ]:
print("Flights sample")
flights.show(5, truncate=False)

print("Airports sample")
airports.show(5, truncate=False)

print("Airlines sample")
airlines.show(5, truncate=False)

### Conclusion

- All three typed Parquet datasets were read successfully.
- The row counts match the results of the previous validation step:
  - flights: 5,819,079
  - airports: 322
  - airlines: 14
- The schema is typed, so the data is ready for the next data quality profiling step.

## 3. Completeness: Null and Blank Values

This section measures data completeness in two forms:

- **Null**: a value missing at the schema level;
- **Blank**: an empty string or a string containing only whitespace.

These are checked separately to avoid conflating nulls with missing text values.


In [ ]:
from pyspark.sql import types as T

def completeness_profile(df):
    total_rows = df.count()

    null_exprs = [
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]

    null_counts = df.agg(*null_exprs).first().asDict()

    string_cols = [
        field.name
        for field in df.schema.fields
        if isinstance(field.dataType, T.StringType)
    ]

    if string_cols:
        blank_exprs = [
            F.sum(
                (
                    F.col(c).isNotNull()
                    & (F.trim(F.col(c)) == "")
                ).cast("int")
            ).alias(c)
            for c in string_cols
        ]

        blank_counts = df.agg(*blank_exprs).first().asDict()
    else:
        blank_counts = {}

    rows = []

    for c in df.columns:
        null_count = int(null_counts[c] or 0)
        blank_count = int(blank_counts.get(c, 0) or 0)

        rows.append(
            (
                c,
                null_count,
                round(null_count / total_rows * 100, 4),
                blank_count,
                round(blank_count / total_rows * 100, 4),
            )
        )

    return spark.createDataFrame(
        rows,
        [
            "column",
            "null_count",
            "null_rate_pct",
            "blank_count",
            "blank_rate_pct",
        ],
    )

In [ ]:
flights_completeness = completeness_profile(flights)

flights_completeness.orderBy(
    F.desc("null_rate_pct")
).show(31, truncate=False)

In [ ]:
airports_completeness = completeness_profile(airports)

airports_completeness.orderBy(
    F.desc("null_rate_pct")
).show(truncate=False)

In [ ]:
airlines_completeness = completeness_profile(airlines)

airlines_completeness.show(truncate=False)

### Conclusion

- `ARRIVAL_DELAY` is missing in 105,071 rows, about 1.81% of all flights.
- The delay-cause columns are missing about 81.7% of their values; this missingness has business meaning and is also post-outcome information.
- `LATITUDE` and `LONGITUDE` are both missing in 3 airport rows.
- No significant blank/whitespace values were found in the typed raw data.

These missing values **are not handled in this notebook**; they are only recorded here for handoff to the appropriate module.

## 4. Missingness Patterns

After identifying which columns have missing values, this section checks whether those missing values follow a clear pattern.

Two main patterns are examined:

1. Whether null `ARRIVAL_DELAY` values are associated with `CANCELLED` or `DIVERTED` flights.
2. Whether delay-cause columns are populated only when a flight has been significantly delayed.

In [ ]:
arrival_missing_pattern = flights.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.col("ARRIVAL_DELAY").isNull().cast("int")
    ).alias("arrival_delay_null"),

    F.sum(
        (F.col("CANCELLED") == 1).cast("int")
    ).alias("cancelled"),

    F.sum(
        (F.col("DIVERTED") == 1).cast("int")
    ).alias("diverted"),

    F.sum(
        (
            F.col("ARRIVAL_DELAY").isNull()
            & (F.col("CANCELLED") == 0)
            & (F.col("DIVERTED") == 0)
        ).cast("int")
    ).alias("arrival_null_but_not_cancelled_or_diverted"),
)

arrival_missing_pattern.show(truncate=False)

### Conclusion: `ARRIVAL_DELAY`

- 105,071 rows are missing `ARRIVAL_DELAY`.
- All of these rows belong to flights that were `CANCELLED` or `DIVERTED`.
- There are no null `ARRIVAL_DELAY` values outside these two groups.

Therefore, this is not random missingness. It is tied to the flight's business status and directly affects the definition of **eligible rows** for both modeling tasks.

### Delay-Cause Columns

The delay-cause columns describe the reasons for delays and have a very high missing rate.

This section checks whether non-null values occur primarily when
`ARRIVAL_DELAY >= 15`.

In [ ]:
delay_cause_cols = [
    "AIR_SYSTEM_DELAY",
    "SECURITY_DELAY",
    "AIRLINE_DELAY",
    "LATE_AIRCRAFT_DELAY",
    "WEATHER_DELAY",
]

delay_pattern_exprs = []

for c in delay_cause_cols:
    delay_pattern_exprs.extend([
        F.sum(
            F.col(c).isNotNull().cast("int")
        ).alias(f"{c}_nonnull"),

        F.sum(
            (
                F.col(c).isNotNull()
                & (F.col("ARRIVAL_DELAY") < 15)
            ).cast("int")
        ).alias(f"{c}_nonnull_when_arrival_delay_lt15"),
    ])

delay_pattern = flights.agg(*delay_pattern_exprs).first().asDict()

In [ ]:
delay_pattern_rows = []

for c in delay_cause_cols:
    nonnull = delay_pattern[f"{c}_nonnull"]
    below_15 = delay_pattern[f"{c}_nonnull_when_arrival_delay_lt15"]

    delay_pattern_rows.append(
        (c, nonnull, below_15)
    )

delay_pattern_df = spark.createDataFrame(
    delay_pattern_rows,
    [
        "column",
        "nonnull_count",
        "nonnull_when_arrival_delay_lt_15",
    ],
)

delay_pattern_df.show(truncate=False)

### Conclusion: Delay-Cause Columns

- The delay-cause columns are missing about 81.7% of their values.
- Non-null values occur only when `ARRIVAL_DELAY >= 15`.
- This missingness has business meaning: delay causes are recorded only when a flight is delayed enough to warrant breaking down the reasons.
- These columns are also **post-outcome leakage**, because they are known only after the delay has occurred.

Therefore, nulls should not be automatically filled with 0 during data preparation.

## 5. Uniqueness and Duplicates

This section checks the uniqueness of lookup keys and the duplicate status.

- `airports.IATA_CODE` must be a unique key in the airport table.
- `airlines.IATA_CODE` must be a unique key in the airline table.
- Exact full-row duplicates in `flights` were already checked during raw data profiling; this notebook does not rerun that expensive check unnecessarily.

In [ ]:
def lookup_uniqueness(df, key_col):
    result = df.agg(
        F.count("*").alias("rows"),
        F.countDistinct(key_col).alias("distinct_keys"),
        F.sum(
            (
                F.col(key_col).isNull()
                | (F.trim(F.col(key_col)) == "")
            ).cast("int")
        ).alias("missing_keys"),
    ).first()

    duplicate_keys = result["rows"] - result["distinct_keys"]

    return (
        result["rows"],
        result["distinct_keys"],
        result["missing_keys"],
        duplicate_keys,
    )

In [ ]:
airports_stats = lookup_uniqueness(airports, "IATA_CODE")
airlines_stats = lookup_uniqueness(airlines, "IATA_CODE")

uniqueness_rows = [
    ("airports", *airports_stats),
    ("airlines", *airlines_stats),
]

uniqueness_df = spark.createDataFrame(
    uniqueness_rows,
    [
        "dataset",
        "rows",
        "distinct_keys",
        "missing_keys",
        "duplicate_keys",
    ]
)

uniqueness_df.show(truncate=False)

### Conclusion

- Neither `airports.IATA_CODE` nor `airlines.IATA_CODE` contains missing or duplicate keys.
- Both lookup tables meet the uniqueness requirement for joins.
- Exact full-row duplicates in `flights` are not checked again in this notebook; their status is explicitly recorded as `SKIPPED` because the raw data was profiled earlier.

## 6. Validity and Domain Checks

This section checks whether values fall within valid domains.

Main rules:

- `YEAR = 2015`
- `MONTH ∈ [1, 12]`
- `DAY ∈ [1, 31]`
- `DAY_OF_WEEK ∈ [1, 7]`
- `CANCELLED ∈ {0, 1}`
- `DIVERTED ∈ {0, 1}`
- `LATITUDE ∈ [-90, 90]`
- `LONGITUDE ∈ [-180, 180]`

In [ ]:
domain_checks = [
    ("YEAR", ~F.col("YEAR").isin([2015])),
    ("MONTH", ~F.col("MONTH").between(1, 12)),
    ("DAY", ~F.col("DAY").between(1, 31)),
    ("DAY_OF_WEEK", ~F.col("DAY_OF_WEEK").between(1, 7)),
    ("CANCELLED", ~F.col("CANCELLED").isin([0, 1])),
    ("DIVERTED", ~F.col("DIVERTED").isin([0, 1])),
]

domain_results = []

for column, invalid_condition in domain_checks:
    invalid_count = flights.filter(
        F.col(column).isNotNull() & invalid_condition
    ).count()

    domain_results.append(
        (
            column,
            invalid_count,
            "PASS" if invalid_count == 0 else "FAIL",
        )
    )

domain_results_df = spark.createDataFrame(
    domain_results,
    ["column", "invalid_count", "status"]
)

domain_results_df.show(truncate=False)

In [ ]:
coordinate_check = airports.agg(
    F.sum(
        (
            F.col("LATITUDE").isNotNull()
            & ~F.col("LATITUDE").between(-90, 90)
        ).cast("int")
    ).alias("invalid_latitude"),

    F.sum(
        (
            F.col("LONGITUDE").isNotNull()
            & ~F.col("LONGITUDE").between(-180, 180)
        ).cast("int")
    ).alias("invalid_longitude"),

    F.sum(
        (
            F.col("LATITUDE").isNull()
            & F.col("LONGITUDE").isNull()
        ).cast("int")
    ).alias("both_coordinates_null"),

    F.sum(
        (
            F.col("LATITUDE").isNull()
            != F.col("LONGITUDE").isNull()
        ).cast("int")
    ).alias("only_one_coordinate_null"),
)

coordinate_check.show(truncate=False)

### Conclusion

- No out-of-domain values were found in the checked date and status columns.
- No latitude or longitude values fall outside the valid geographic ranges.
- Three airports are missing both `LATITUDE` and `LONGITUDE`.
- There are no cases where only one of the two coordinates is missing.

## 7. HHMM Validity

The time columns are currently retained as strings in HHMM format.

This section checks:

- null values;
- blank strings;
- valid values in the range `0000–2359`;
- the special value `2400`;
- invalid formats or ranges.

The notebook checks validity only and does not convert values to timestamps.

In [ ]:
hhmm_cols = [
    "SCHEDULED_DEPARTURE",
    "DEPARTURE_TIME",
    "WHEELS_OFF",
    "WHEELS_ON",
    "SCHEDULED_ARRIVAL",
    "ARRIVAL_TIME",
]

In [ ]:
def hhmm_profile(df, column):
    c = F.col(column)

    valid_format = c.rlike(r"^\d{4}$")

    hour = c.substr(1, 2).cast("int")
    minute = c.substr(3, 2).cast("int")

    valid_regular_time = (
        valid_format
        & hour.between(0, 23)
        & minute.between(0, 59)
    )

    is_2400 = c == "2400"

    result = df.agg(
        F.count("*").alias("rows"),

        F.sum(c.isNull().cast("int")).alias("null_count"),

        F.sum(
            (
                c.isNotNull()
                & (F.trim(c) == "")
            ).cast("int")
        ).alias("blank_count"),

        F.sum(
            valid_regular_time.cast("int")
        ).alias("valid_hhmm_count"),

        F.sum(
            is_2400.cast("int")
        ).alias("value_2400_count"),

        F.sum(
            (
                c.isNotNull()
                & (F.trim(c) != "")
                & ~valid_regular_time
                & ~is_2400
            ).cast("int")
        ).alias("invalid_count"),
    ).first()

    return (
        column,
        result["null_count"],
        result["blank_count"],
        result["valid_hhmm_count"],
        result["value_2400_count"],
        result["invalid_count"],
    )

In [ ]:
hhmm_results = [
    hhmm_profile(flights, c)
    for c in hhmm_cols
]

hhmm_df = spark.createDataFrame(
    hhmm_results,
    [
        "column",
        "null_count",
        "blank_count",
        "valid_hhmm_count",
        "value_2400_count",
        "invalid_count",
    ],
)

hhmm_df.show(truncate=False)

### Conclusion

- No HHMM values with an invalid format or out-of-range value were found in the six checked time columns.
- The value `2400` is recorded separately and is not treated as an error under the dataset convention.
- Actual-time columns may contain nulls due to flight status; this is a completeness/business-state issue, not a format error.

The time columns remain strings in the typed raw data.

## 8. Cardinality

Cardinality is the number of distinct values in a column.

This section focuses on important categorical/identifier columns to identify:

- low cardinality: suitable for common encoding approaches;
- high cardinality: requires careful consideration during feature engineering.

The notebook measures cardinality but does not perform encoding.

In [ ]:
cardinality_cols = [
    "AIRLINE",
    "FLIGHT_NUMBER",
    "TAIL_NUMBER",
    "ORIGIN_AIRPORT",
    "DESTINATION_AIRPORT",
    "CANCELLATION_REASON",
]

In [ ]:
cardinality_exprs = [
    F.countDistinct(c).alias(c)
    for c in cardinality_cols
]

cardinality_stats = (
    flights
    .agg(*cardinality_exprs)
    .first()
    .asDict()
)

In [ ]:
cardinality_rows = [
    (column, cardinality_stats[column])
    for column in cardinality_cols
]

cardinality_df = spark.createDataFrame(
    cardinality_rows,
    ["column", "distinct_count"]
)

cardinality_df.orderBy(
    F.desc("distinct_count")
).show(truncate=False)

### Conclusion

- `AIRLINE` has low cardinality, with only a small number of airlines.
- `ORIGIN_AIRPORT` and `DESTINATION_AIRPORT` have moderate cardinality.
- `FLIGHT_NUMBER` and especially `TAIL_NUMBER` have substantially higher cardinality.
- Therefore, `TAIL_NUMBER` requires further review before being used as a feature.
- Cardinality is recorded for handoff to the feature engineering module; this notebook does not perform encoding.

## 9. Referential Integrity and Join Coverage

This section checks the match rate between the `flights` table and the two lookup tables:

- `flights.AIRLINE` → `airlines.IATA_CODE`
- `flights.ORIGIN_AIRPORT` → `airports.IATA_CODE`
- `flights.DESTINATION_AIRPORT` → `airports.IATA_CODE`

The goal is to measure match/unmatch rates and identify the main reason why some airport codes do not match directly.

The notebook checks coverage only; it does not map DOT codes to IATA codes.

In [ ]:
airline_join = flights.join(
    airlines.select(
        F.col("IATA_CODE").alias("AIRLINE_KEY")
    ),
    flights["AIRLINE"] == F.col("AIRLINE_KEY"),
    "left"
)

airline_coverage = airline_join.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.col("AIRLINE_KEY").isNotNull().cast("int")
    ).alias("matched_rows"),

    F.sum(
        F.col("AIRLINE_KEY").isNull().cast("int")
    ).alias("unmatched_rows"),
).first()

airline_coverage_pct = (
    airline_coverage["matched_rows"]
    / airline_coverage["total_rows"]
    * 100
)

print("Airline lookup coverage")
print("-" * 40)
print(f"Total rows   : {airline_coverage['total_rows']:,}")
print(f"Matched rows : {airline_coverage['matched_rows']:,}")
print(f"Unmatched    : {airline_coverage['unmatched_rows']:,}")
print(f"Coverage     : {airline_coverage_pct:.4f}%")

In [ ]:
def airport_coverage(df, airport_col, airports_df):
    joined = df.join(
        airports_df.select(
            F.col("IATA_CODE").alias("AIRPORT_KEY")
        ),
        df[airport_col] == F.col("AIRPORT_KEY"),
        "left"
    )

    result = joined.agg(
        F.count("*").alias("total_rows"),

        F.sum(
            F.col("AIRPORT_KEY").isNotNull().cast("int")
        ).alias("matched_rows"),

        F.sum(
            F.col("AIRPORT_KEY").isNull().cast("int")
        ).alias("unmatched_rows"),
    ).first()

    coverage_pct = (
        result["matched_rows"]
        / result["total_rows"]
        * 100
    )

    return (
        result["total_rows"],
        result["matched_rows"],
        result["unmatched_rows"],
        coverage_pct,
    )

In [ ]:
origin_stats = airport_coverage(
    flights,
    "ORIGIN_AIRPORT",
    airports
)

destination_stats = airport_coverage(
    flights,
    "DESTINATION_AIRPORT",
    airports
)

airport_coverage_df = spark.createDataFrame(
    [
        ("ORIGIN_AIRPORT", *origin_stats),
        ("DESTINATION_AIRPORT", *destination_stats),
    ],
    [
        "column",
        "total_rows",
        "matched_rows",
        "unmatched_rows",
        "coverage_pct",
    ]
)

airport_coverage_df.show(truncate=False)

In [ ]:
numeric_airport_by_month = (
    flights
    .groupBy("MONTH")
    .agg(
        F.sum(
            F.col("ORIGIN_AIRPORT")
            .rlike(r"^\d{5}$")
            .cast("int")
        ).alias("numeric_origin_rows"),

        F.sum(
            F.col("DESTINATION_AIRPORT")
            .rlike(r"^\d{5}$")
            .cast("int")
        ).alias("numeric_destination_rows"),
    )
    .orderBy("MONTH")
)

numeric_airport_by_month.show(12, truncate=False)

In [ ]:
origin_unmatched_codes = (
    flights
    .join(
        airports.select(
            F.col("IATA_CODE").alias("AIRPORT_KEY")
        ),
        flights["ORIGIN_AIRPORT"] == F.col("AIRPORT_KEY"),
        "left"
    )
    .filter(F.col("AIRPORT_KEY").isNull())
    .select("ORIGIN_AIRPORT")
    .distinct()
    .count()
)

destination_unmatched_codes = (
    flights
    .join(
        airports.select(
            F.col("IATA_CODE").alias("AIRPORT_KEY")
        ),
        flights["DESTINATION_AIRPORT"] == F.col("AIRPORT_KEY"),
        "left"
    )
    .filter(F.col("AIRPORT_KEY").isNull())
    .select("DESTINATION_AIRPORT")
    .distinct()
    .count()
)

print("Distinct unmatched airport codes")
print("-" * 40)
print(f"Origin      : {origin_unmatched_codes}")
print(f"Destination : {destination_unmatched_codes}")

### Conclusion

- `AIRLINE` matches the `airlines` table at 100%.
- `ORIGIN_AIRPORT` and `DESTINATION_AIRPORT` each match the `airports` table directly at only about 91.65%.
- Most unmatched values are concentrated in October, when numeric five-digit DOT airport codes appear in the data.
- The `airports` table uses only three-character IATA codes, so these DOT codes cannot be joined directly.

Therefore, referential integrity for the airline lookup is good, but airport lookup coverage is **partial**. The Custom/Feature module needs to apply a verified DOT → IATA mapping before calculating Haversine distance.

## 10. Target and Eligibility

Both tasks use the same original target, `ARRIVAL_DELAY`.

A row is considered **eligible** only when:

- `ARRIVAL_DELAY` is not null;
- `CANCELLED = 0`;
- `DIVERTED = 0`.

This section identifies and counts eligible rows; it does not create a train/test split or model dataset.

In [ ]:
eligible_condition = (
    F.col("ARRIVAL_DELAY").isNotNull()
    & (F.col("CANCELLED") == 0)
    & (F.col("DIVERTED") == 0)
)

eligibility_stats = flights.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        eligible_condition.cast("int")
    ).alias("eligible_rows"),

    F.sum(
        (~eligible_condition).cast("int")
    ).alias("excluded_rows"),

    F.sum(
        F.col("ARRIVAL_DELAY").isNull().cast("int")
    ).alias("arrival_delay_null"),

    F.sum(
        (F.col("CANCELLED") == 1).cast("int")
    ).alias("cancelled"),

    F.sum(
        (F.col("DIVERTED") == 1).cast("int")
    ).alias("diverted"),
).first()

eligibility_df = spark.createDataFrame(
    [(
        eligibility_stats["total_rows"],
        eligibility_stats["eligible_rows"],
        eligibility_stats["excluded_rows"],
        eligibility_stats["arrival_delay_null"],
        eligibility_stats["cancelled"],
        eligibility_stats["diverted"],
    )],
    [
        "total_rows",
        "eligible_rows",
        "excluded_rows",
        "arrival_delay_null",
        "cancelled",
        "diverted",
    ]
)

eligibility_df.show(truncate=False)

### Conclusion

- 5,714,008 flights are eligible for both tasks.
- Most excluded rows are `CANCELLED` or `DIVERTED` flights, for which the target `ARRIVAL_DELAY` is null.



## 11. Classification Label Distribution

The classification task uses a provisional label:

- `1` if `ARRIVAL_DELAY > 30`;
- `0` if `ARRIVAL_DELAY <= 30`.

In [ ]:
eligible_flights = flights.filter(eligible_condition)

label_distribution = (
    eligible_flights
    .select(
        F.when(
            F.col("ARRIVAL_DELAY") > 30, 1
        ).otherwise(0).alias("label")
    )
    .groupBy("label")
    .count()
    .orderBy("label")
)

label_distribution.show()

In [ ]:
label_counts = {
    row["label"]: row["count"]
    for row in label_distribution.collect()
}

negative = label_counts.get(0, 0)
positive = label_counts.get(1, 0)
total = negative + positive

positive_rate = positive / total * 100

print("Classification label distribution")
print("-" * 45)
print(f"Negative (<=30 min) : {negative:,}")
print(f"Positive (>30 min)  : {positive:,}")
print(f"Positive rate       : {positive_rate:.4f}%")

### Conclusion

- The positive class accounts for only about 11.07% of rows.
- The classification task has a significant class imbalance.

## 12. Numeric Descriptive Statistics

This section describes several important numeric variables to understand the data range, center, and spread.

Outliers are not handled in this notebook.

In [ ]:
numeric_cols = [
    "ARRIVAL_DELAY",
    "DEPARTURE_DELAY",
    "SCHEDULED_TIME",
    "DISTANCE",
]

numeric_profile = (
    flights
    .select(numeric_cols)
    .summary(
        "count",
        "min",
        "mean",
        "stddev",
        "max"
    )
)

numeric_profile.show(truncate=False)

In [ ]:
quantile_rows = []

for column in numeric_cols:
    q25, q50, q75 = flights.approxQuantile(
        column,
        [0.25, 0.50, 0.75],
        0.01
    )

    quantile_rows.append(
        (column, q25, q50, q75)
    )

quantiles_df = spark.createDataFrame(
    quantile_rows,
    ["column", "q25", "median", "q75"]
)

quantiles_df.show(truncate=False)

In [ ]:
arrival_stats = eligible_flights.agg(
    F.min("ARRIVAL_DELAY").alias("min"),
    F.max("ARRIVAL_DELAY").alias("max"),
    F.avg("ARRIVAL_DELAY").alias("mean"),
    F.stddev("ARRIVAL_DELAY").alias("stddev"),

    F.sum(
        (F.col("ARRIVAL_DELAY") < 0).cast("int")
    ).alias("negative_rows"),

    F.count("*").alias("rows"),
).first()

negative_pct = (
    arrival_stats["negative_rows"]
    / arrival_stats["rows"]
    * 100
)

print("ARRIVAL_DELAY profile")
print("-" * 40)
print(f"Min              : {arrival_stats['min']}")
print(f"Max              : {arrival_stats['max']}")
print(f"Mean             : {arrival_stats['mean']:.4f}")
print(f"Negative rows    : {arrival_stats['negative_rows']:,}")
print(f"Negative rate    : {negative_pct:.4f}%")

### Conclusion

- `ARRIVAL_DELAY` has both negative and positive values.
- Negative values are valid in this context: the flight arrived earlier than scheduled.
- About 61% of target values are negative.
- Therefore, negative target values must not be clipped during data preparation.
- The Model module should separately consider models/losses that require positive targets.

## 13. Leakage and Inference Contract

The project's chosen inference time is:

**before the flight actually departs.**

Therefore, only information known before actual departure may be used. This section classifies columns by role to prevent data leakage.

In [ ]:
feature_roles = [
    ("YEAR", "Candidate feature", "Pre-departure"),
    ("MONTH", "Candidate feature", "Pre-departure"),
    ("DAY", "Candidate feature", "Pre-departure"),
    ("DAY_OF_WEEK", "Candidate feature", "Pre-departure"),
    ("AIRLINE", "Candidate feature", "Pre-departure"),
    ("FLIGHT_NUMBER", "Candidate feature", "Pre-departure"),
    ("ORIGIN_AIRPORT", "Candidate feature", "Pre-departure"),
    ("DESTINATION_AIRPORT", "Candidate feature", "Pre-departure"),
    ("SCHEDULED_DEPARTURE", "Candidate feature", "Pre-departure"),
    ("SCHEDULED_ARRIVAL", "Candidate feature", "Pre-departure"),
    ("SCHEDULED_TIME", "Candidate feature", "Pre-departure"),
    ("DISTANCE", "Candidate feature", "Pre-departure"),

    ("TAIL_NUMBER", "Requires review", "Availability/high cardinality"),

    ("ARRIVAL_DELAY", "Target", "Outcome"),

    ("CANCELLED", "Eligibility only", "Not feature"),
    ("DIVERTED", "Eligibility only", "Not feature"),

    ("DEPARTURE_TIME", "Leakage", "Post-departure"),
    ("DEPARTURE_DELAY", "Leakage", "Post-departure"),
    ("TAXI_OUT", "Leakage", "Post-departure"),
    ("WHEELS_OFF", "Leakage", "Post-departure"),
    ("ELAPSED_TIME", "Leakage", "Post-departure"),
    ("AIR_TIME", "Leakage", "Post-departure"),
    ("WHEELS_ON", "Leakage", "Post-departure"),
    ("TAXI_IN", "Leakage", "Post-departure"),
    ("ARRIVAL_TIME", "Leakage", "Post-outcome"),
    ("CANCELLATION_REASON", "Leakage", "Post-outcome"),
    ("AIR_SYSTEM_DELAY", "Leakage", "Post-outcome"),
    ("SECURITY_DELAY", "Leakage", "Post-outcome"),
    ("AIRLINE_DELAY", "Leakage", "Post-outcome"),
    ("LATE_AIRCRAFT_DELAY", "Leakage", "Post-outcome"),
    ("WEATHER_DELAY", "Leakage", "Post-outcome"),
]

feature_roles_df = spark.createDataFrame(
    feature_roles,
    ["column", "role", "availability"]
)

feature_roles_df.show(40, truncate=False)

In [ ]:
missing_decisions = [
    (
        "ARRIVAL_DELAY",
        "105,071 null rows",
        "No target for regression/classification",
        "Apply the eligibility rule; do not impute the target",
        "Pipeline / Model",
    ),
    (
        "LATITUDE / LONGITUDE",
        "3 airport rows missing both coordinates",
        "Affects airport features and Haversine distance",
        "Verify or supplement in the feature module",
        "Custom/Feature",
    ),
    (
        "Delay-cause columns",
        "About 81.7% null",
        "Business-related missingness; also leakage",
        "Do not automatically fill with 0; exclude from the pre-departure feature set",
        "Custom/Feature",
    ),
    (
        "TAIL_NUMBER",
        "Has missing values and high cardinality",
        "May affect encoding/feature stability",
        "Review availability and feature usefulness",
        "Custom/Feature",
    ),
    (
        "Actual-time columns",
        "Missingness depends on flight status",
        "Not suitable for inference before departure",
        "Do not use as a pre-departure feature",
        "Pipeline",
    ),
]

missing_decision_df = spark.createDataFrame(
    missing_decisions,
    [
        "column_or_group",
        "finding",
        "impact",
        "recommended_next_action",
        "owner",
    ],
)

missing_decision_df.show(truncate=False)

## 15. Data Quality Scorecard

The scorecard summarizes data quality status across the main dimensions.

Not every dimension is required to PASS.

In [ ]:
scorecard = [
    (
        "Completeness",
        "PARTIAL",
        "Target, coordinates, and some post-outcome columns have missing values",
        "Handle each group separately in a later module",
    ),
    (
        "Validity",
        "PASS",
        "No domain, HHMM, or coordinate-range violations found",
        "Keep validation in the pipeline",
    ),
    (
        "Uniqueness",
        "PARTIAL",
        "Lookup keys are unique; full-row duplicate check was not rerun in this notebook",
        "Rerun the exact duplicate check if a full audit is needed",
    ),
    (
        "Consistency",
        "PASS",
        "Target missingness is consistent with the cancelled/diverted pattern",
        "Preserve business invariants in later processing",
    ),
    (
        "Referential Integrity",
        "PARTIAL",
        "Airline 100%; airport lookup about 91.65%",
        "A verified DOT → IATA mapping is needed",
    ),
    (
        "Accuracy",
        "UNASSESSED",
        "No external ground truth is available at the record level",
        "Absolute accuracy cannot be confirmed",
    ),
    (
        "Timeliness",
        "LIMITATION",
        "The dataset is from 2015",
        "Does not directly represent current airline operations",
    ),
]

scorecard_df = spark.createDataFrame(
    scorecard,
    [
        "dimension",
        "status",
        "evidence",
        "limitation_or_next_action",
    ],
)

scorecard_df.show(truncate=False)

## 16. Data Quality Summary

### Main conclusions

- Typed Parquet was successfully read and profiled using PySpark.
- The schema and basic domains are valid.
- Missing `ARRIVAL_DELAY` values follow a clear pattern associated with cancelled/diverted flights.
- The classification target is imbalanced, with a positive rate of about 11%.
- Airport lookup matches only about 91.65% because numeric DOT codes appear in October.
- Some actual/post-outcome columns would cause data leakage if used for inference before departure.
- The dataset is still **typed raw**, not cleaned or model-ready.

### Modeling constraints

- Coordinate enrichment must handle missing coordinates and numeric DOT airport codes.
- Pre-departure features must satisfy the inference and eligibility contracts.
- Model comparisons require identical splits, class imbalance handling, and support for negative regression targets.
- Serving payloads must match the finalized input schema.


In [ ]:
spark.stop()

print("Spark session stopped.")